# POC testing for full UCR dataset
Using resamples from reproduction

In [2]:
import time
import numpy as np
import pandas as pd
from numba import njit
from aeon.datasets import load_from_ts_file
from sklearn.preprocessing import LabelEncoder
import os
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import f1_score

from minirocket_poc import fit, transform

### Initialisation

In [3]:
DATA_DIR = "../datasets"
SPLIT_DIR = "../splits"
RESULTS_DIR = "../results_improvements"
REPRODUCTION_CSV = "../minirocket_ucr109_results/minirocket_ucr109_results/minirocket_per_resample.csv"

RUN_NAME = "poc"

NUM_FEATURES = 10_000
RESAMPLES = range(30)
DATASETS = None             # used for quick testing, None to run for all datasets

CLASSIFIERS = ["ridge"]

VARIANTS = {
    "raw": 0,                  # original MiniRocket
    "poc_2000": 2000,              # PPV features + 2000 co-activation pairs
}

### Loading the data

In [4]:
def load_dataset(name):
    X_train, y_train = load_from_ts_file(f"{DATA_DIR}/{name}/{name}_TRAIN.ts")
    X_test, y_test = load_from_ts_file(f"{DATA_DIR}/{name}/{name}_TEST.ts")

    X_all = np.concatenate([X_train, X_test])[:, 0, :].astype(np.float32)
    y_all = LabelEncoder().fit_transform(np.concatenate([y_train, y_test]))

    return X_all, y_all, len(X_train)



if DATASETS is None:
    dataset_names = sorted(f[:-len("_splits.npz")] for f in os.listdir(SPLIT_DIR) if f.endswith("_splits.npz"))
else:
    dataset_names = list(DATASETS)

data = {}
for name in dataset_names:
    data[name] = load_dataset(name)

print(f"Loaded {len(data)} datasets, {sum(len(y_all) for _, y_all, _ in data.values()):,} series ")

Loaded 109 datasets, 172,051 series 


### Loading the resamples

In [5]:
splits = {}

for name in dataset_names:

    X_all, y_all, n_train_original = data[name]
    original_counts = np.bincount(y_all[:n_train_original])

    splits[name] = {}

    with np.load(f"{SPLIT_DIR}/{name}_splits.npz") as file:

        assert int(file["n_train_original"]) == n_train_original, name

        for r in RESAMPLES:
            train_index = file[f"train_{r:02d}"]
            test_index = file[f"test_{r:02d}"]

            assert np.array_equal(np.sort(np.concatenate([train_index, test_index])), np.arange(len(y_all))), (name, r)

            assert np.array_equal(np.bincount(y_all[train_index], minlength = len(original_counts)), original_counts), (name, r)

            splits[name][r] = (train_index, test_index)

    if 0 in splits[name]:
        assert np.array_equal(splits[name][0][0], np.arange(n_train_original)), name

print(f"Splits OK: {len(splits)} datasets x {len(RESAMPLES)} resamples")

Splits OK: 109 datasets x 30 resamples


### Feature extraction with new POC features

In [6]:
@njit
def seed_numba(seed):
    np.random.seed(seed)


def fit_parameters(X_train, num_pairs, seed):
    seed_numba(seed)
    parameters = fit(X_train, num_features = NUM_FEATURES - num_pairs, num_pairs = num_pairs, seed = seed)

    # series too short for any pair, use the whole budget for PPV features instead
    if num_pairs > 0 and len(parameters[3]) == 0:
        seed_numba(seed)
        parameters = fit(X_train, num_features = NUM_FEATURES, num_pairs = 0, seed = seed)

    return parameters

name = dataset_names[0]
X_all, y_all, n_train_original = data[name]
X_example = X_all[:n_train_original]

for variant, num_pairs in VARIANTS.items():
    parameters = fit_parameters(X_example, num_pairs, seed = 0)
    F = transform(X_example, parameters)
    print(f"{name} | {variant}: {X_example.shape} series -> {F.shape} features ({len(parameters[3])} pairs)")

ACSF1 | raw: (100, 1460) series -> (100, 9996) features (0 pairs)
ACSF1 | poc_2000: (100, 1460) series -> (100, 9980) features (2000 pairs)


### Model training

In [7]:
RIDGE_ALPHAS = np.logspace(-3, 3, 10)


def run_ridge(X_train, y_train, X_test, y_test, num_pairs, seed):
    # fit: choose dilations and biases
    start = time.perf_counter()
    parameters = fit_parameters(X_train, num_pairs, seed)
    fit_seconds = time.perf_counter() - start

    # transform: series -> features
    start = time.perf_counter()
    F_train = transform(X_train, parameters)
    F_test = transform(X_test, parameters)
    transform_seconds = time.perf_counter() - start

    # training: normalise, fit the classifier, score the test set
    start = time.perf_counter()
    feature_mean = F_train.mean(axis = 0)
    F_train -= feature_mean
    F_test -= feature_mean
    feature_scale = np.sqrt((F_train ** 2).sum(axis = 0))
    feature_scale[feature_scale == 0] = 1.0
    F_train /= feature_scale
    F_test /= feature_scale

    model = RidgeClassifierCV(alphas = RIDGE_ALPHAS)
    model.fit(F_train, y_train)
    predictions = model.predict(F_test)
    accuracy = (predictions == y_test).mean()
    f1 = f1_score(y_test, predictions, average = "macro", zero_division = 0)
    train_seconds = time.perf_counter() - start

    return {
        "accuracy": accuracy,
        "f1": f1,
        "fit_seconds": fit_seconds,
        "transform_seconds": transform_seconds,
        "train_seconds": train_seconds,
        "pairs_used": len(parameters[3])
    }


PER_RESAMPLE_CSV = f"{RESULTS_DIR}/{RUN_NAME}_ucr109_per_resample.csv"

try:
    saved = pd.read_csv(PER_RESAMPLE_CSV)
    rows = saved.to_dict("records")
    print(f"Continuing: {len(rows)} results already in {PER_RESAMPLE_CSV}")
except FileNotFoundError:
    rows = []

done = {(row["dataset"], row["resample"], row["variant"], row["classifier"]) for row in rows}

run_start = time.perf_counter()

for dataset_number, name in enumerate(dataset_names, start = 1):

    X_all, y_all, _ = data[name]

    for r in RESAMPLES:

        train_index, test_index = splits[name][r]

        X_train, y_train = X_all[train_index], y_all[train_index]
        X_test, y_test = X_all[test_index], y_all[test_index]

        for variant, num_pairs in VARIANTS.items():
            for classifier in CLASSIFIERS:

                if (name, r, variant, classifier) in done:
                    continue

                start = time.perf_counter()

                if classifier == "ridge":
                    outcome = run_ridge(X_train, y_train, X_test, y_test, num_pairs, seed = r)
                else:
                    raise ValueError(f"Unknown classifier: {classifier}")

                rows.append({
                    "dataset": name,
                    "resample": r,
                    "variant": variant,
                    "classifier": classifier,
                    **outcome,
                    "total_seconds": time.perf_counter() - start
                })

        pd.DataFrame(rows).to_csv(PER_RESAMPLE_CSV, index = False)

    means = " | ".join(
        f"{classifier} {variant} " + "{:.4f}".format(np.mean([
            row["accuracy"] for row in rows
            if row["dataset"] == name and row["variant"] == variant and row["classifier"] == classifier
        ]))
        for classifier in CLASSIFIERS
        for variant in VARIANTS
    )
    print(f"{dataset_number:3d}/{len(dataset_names)} {name:30} {means} | {(time.perf_counter() - run_start) / 60:6.1f} min")

print(f"\nFinished. Per-resample results: {PER_RESAMPLE_CSV}")

  1/109 ACSF1                          ridge raw 0.8237 | ridge poc_2000 0.8250 |    1.5 min
  2/109 Adiac                          ridge raw 0.8084 | ridge poc_2000 0.8161 |    2.2 min
  3/109 ArrowHead                      ridge raw 0.8800 | ridge poc_2000 0.8838 |    2.5 min
  4/109 BME                            ridge raw 0.9922 | ridge poc_2000 0.9964 |    2.7 min
  5/109 Beef                           ridge raw 0.7844 | ridge poc_2000 0.7778 |    2.9 min
  6/109 BeetleFly                      ridge raw 0.9033 | ridge poc_2000 0.9033 |    3.1 min
  7/109 BirdChicken                    ridge raw 0.8967 | ridge poc_2000 0.9000 |    3.3 min
  8/109 CBF                            ridge raw 0.9971 | ridge poc_2000 0.9970 |    3.9 min
  9/109 Car                            ridge raw 0.9200 | ridge poc_2000 0.9156 |    4.4 min
 10/109 Chinatown                      ridge raw 0.9655 | ridge poc_2000 0.9678 |    4.7 min
 11/109 ChlorineConcentration          ridge raw 0.7539 | ridge poc_20

### Results Output

In [8]:
SUMMARY_CSV = f"{RESULTS_DIR}/{RUN_NAME}_summary.csv"
BASELINE = "raw"

results = pd.read_csv(PER_RESAMPLE_CSV)

classifiers = results["classifier"].unique().tolist()
variants = results["variant"].unique().tolist()
others = [variant for variant in variants if variant != BASELINE]

results["run"] = results["classifier"] + "_" + results["variant"]
accuracy = results.pivot(index = ["dataset", "resample"], columns = "run", values = "accuracy")
accuracy = accuracy[[f"{classifier}_{variant}" for classifier in classifiers for variant in variants]]

reproduction = pd.read_csv(REPRODUCTION_CSV).set_index(["dataset", "resample"])["accuracy"]
accuracy["reproduction"] = reproduction.reindex(accuracy.index)

summary = accuracy.groupby("dataset").mean()
for classifier in classifiers:
    for variant in others:
        summary[f"{classifier}_{variant}_minus_{BASELINE}"] = summary[f"{classifier}_{variant}"] - summary[f"{classifier}_{BASELINE}"]

summary["resamples"] = accuracy.groupby("dataset").size()

time_columns = ["fit_seconds", "transform_seconds", "train_seconds", "total_seconds"]
timing = results.pivot_table(index = "dataset", columns = "run", values = time_columns, aggfunc = "mean")
timing.columns = [f"{run}_{step}" for step, run in timing.columns]
summary = summary.join(timing)

f1 = results.pivot(index = ["dataset", "resample"], columns = "run", values = "f1").groupby("dataset").mean()
f1.columns = [f"{run}_f1" for run in f1.columns]
summary = summary.join(f1)

summary.to_csv(SUMMARY_CSV)

print(f"Datasets: {len(summary)} | resamples per dataset: {summary['resamples'].min()} to {summary['resamples'].max()}")
print(f"Mean accuracy over datasets | reproduction: {summary['reproduction'].mean():.4f}")

for classifier in classifiers:
    print(f"\n{classifier}")
    print(f"  {BASELINE:12} {summary[f'{classifier}_{BASELINE}'].mean():.4f}")
    print(f"  {'':12} F1 {summary[f'{classifier}_{BASELINE}_f1'].mean():.4f}")
    for variant in others:
        difference = summary[f"{classifier}_{variant}_minus_{BASELINE}"]
        wins = int((difference > 1e-9).sum())
        losses = int((difference < -1e-9).sum())
        draws = int((difference.abs() <= 1e-9).sum())
        print(f"  {variant:12} {summary[f'{classifier}_{variant}'].mean():.4f} | vs {BASELINE}: {difference.mean():+.4f} " f"| wins / draws / losses: {wins} / {draws} / {losses}")
        f1_difference = summary[f"{classifier}_{variant}_f1"] - summary[f"{classifier}_{BASELINE}_f1"]
        print(f"  {'':12} F1 {summary[f'{classifier}_{variant}_f1'].mean():.4f} | vs {BASELINE}: {f1_difference.mean():+.4f}")


print(f"\nSummary saved to {SUMMARY_CSV}")

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
summary.round(4)

Datasets: 109 | resamples per dataset: 30 to 30
Mean accuracy over datasets | reproduction: 0.8724

ridge
  raw          0.8723
               F1 0.8490
  poc_2000     0.8757 | vs raw: +0.0034 | wins / draws / losses: 72 / 10 / 27
               F1 0.8527 | vs raw: +0.0037

Summary saved to ../results_improvements/poc_summary.csv


,ridge_raw,ridge_poc_2000,reproduction,ridge_poc_2000_minus_raw,resamples,ridge_poc_2000_fit_seconds,ridge_raw_fit_seconds,ridge_poc_2000_total_seconds,ridge_raw_total_seconds,ridge_poc_2000_train_seconds,ridge_raw_train_seconds,ridge_poc_2000_transform_seconds,ridge_raw_transform_seconds,ridge_poc_2000_f1,ridge_raw_f1
dataset,,,,,,,,,,,,,,,
ACSF1,0.8237,0.8250,0.8247,0.0013,30,0.1476,0.1609,2.5423,0.3604,0.0443,0.0490,2.3494,0.1495,0.8231,0.8218
Adiac,0.8084,0.8161,0.8086,0.0077,30,0.0255,0.0296,1.2246,0.2774,0.1723,0.1720,1.0241,0.0732,0.8043,0.7962
ArrowHead,0.8800,0.8838,0.8813,0.0038,30,0.0448,0.0519,0.4734,0.1041,0.0206,0.0204,0.4069,0.0307,0.8839,0.8799
BME,0.9922,0.9964,0.9920,0.0042,30,0.0381,0.0454,0.2802,0.0829,0.0206,0.0196,0.2206,0.0172,0.9964,0.9922
Beef,0.7844,0.7778,0.7878,-0.0067,30,0.0746,0.0793,0.3496,0.1194,0.0209,0.0200,0.2538,0.0198,0.7704,0.7779
BeetleFly,0.9033,0.9033,0.9083,0.0000,30,0.0665,0.0753,0.2523,0.0976,0.0113,0.0110,0.1745,0.0112,0.9024,0.9024
BirdChicken,0.8967,0.9000,0.9017,0.0033,30,0.0645,0.0774,0.2655,0.1003,0.0126,0.0119,0.1884,0.0111,0.8989,0.8955
CBF,0.9971,0.9970,0.9971,-0.0001,30,0.0357,0.0417,1.1186,0.1422,0.0284,0.0282,1.0504,0.0682,0.9970,0.9971
Car,0.9200,0.9156,0.9172,-0.0044,30,0.0793,0.0829,0.7935,0.1583,0.0307,0.0286,0.6828,0.0462,0.9184,0.9223
